# Data Modification

## Quick notes
### What this covers
Changing data: INSERT (all the variants), UPDATE, DELETE, UPSERT, RETURNING, copying data between tables, and changing a table's shape with ALTER TABLE.
### Key points
- `INSERT` adds rows, `UPDATE` changes rows, `DELETE` removes rows. `UPDATE` and `DELETE` without `WHERE` hit every row.
- Always write the `SELECT` with the same `WHERE` first, check the rows, then change `SELECT` to `UPDATE`/`DELETE`.
- UPSERT = `INSERT ... ON CONFLICT DO UPDATE / DO NOTHING`. `excluded.col` is the value you tried to insert.
- `INSERT OR REPLACE` is NOT an update: it deletes the old row and inserts a new one (cascades can fire).
- `RETURNING` gives back the affected rows, so you do not need a second query.
- `INSERT ... SELECT` and `CREATE TABLE ... AS SELECT` copy data in bulk.
- SQLite has no `TRUNCATE`; use `DELETE FROM t`.
- `ALTER TABLE` can ADD / RENAME / DROP a column and RENAME a table, but it cannot change a column's type or constraints.
### Gotchas (the part worth revising twice)
- Forgetting `WHERE` on `UPDATE`/`DELETE`.
- `INSERT OR REPLACE` silently deleting child rows through `ON DELETE CASCADE`.
- `UPDATE ... SET a = b, b = a` uses the OLD values on the right side (a true swap).
- A scalar subquery in `SET` that returns no row sets the column to NULL.
- `changes()` counts rows changed by the LAST statement only; `last_insert_rowid()` is per connection.
- `DROP COLUMN` fails for PRIMARY KEY, UNIQUE, indexed, or referenced columns.
### When it matters
Every real application. Bad UPDATE/DELETE statements are the most common way to lose data in production.
### One-line recap example
```sql
INSERT INTO t(id, qty) VALUES (1, 5)
ON CONFLICT(id) DO UPDATE SET qty = qty + excluded.qty RETURNING *;
```

## 01. Seed data used in this notebook
- A tiny shop: `products`, `customers` and `orders`.
- Prices are stored as whole cents (integers) so there are no rounding surprises.
- See 01-Fundamentals / 05-DDL-And-Integrity/01-Constraints-And-Keys for the table basics.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE products (
    sku    TEXT PRIMARY KEY,
    name   TEXT NOT NULL,
    price  INTEGER NOT NULL CHECK (price >= 0),   -- cents
    stock  INTEGER NOT NULL DEFAULT 0,
    status TEXT NOT NULL DEFAULT 'active'
);
CREATE TABLE customers (
    id    INTEGER PRIMARY KEY,
    name  TEXT NOT NULL,
    email TEXT UNIQUE,
    city  TEXT
);
CREATE TABLE orders (
    id          INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL REFERENCES customers(id) ON DELETE CASCADE,
    sku         TEXT NOT NULL REFERENCES products(sku),
    qty         INTEGER NOT NULL,
    placed_on   TEXT NOT NULL
);
INSERT INTO products (sku, name, price, stock) VALUES
    ('SKU-1', 'Widget', 999, 100),
    ('SKU-2', 'Gadget', 1999, 50),
    ('SKU-3', 'Gizmo', 2999, 0);
INSERT INTO customers (name, email, city) VALUES
    ('Asha', 'asha@example.com', 'Pune'),
    ('Ben', 'ben@example.com', 'Delhi'),
    ('Chitra', NULL, 'Pune');
INSERT INTO orders (customer_id, sku, qty, placed_on) VALUES
    (1, 'SKU-1', 2, '2025-01-05'),
    (1, 'SKU-2', 1, '2025-01-09'),
    (2, 'SKU-1', 5, '2025-02-01');
SELECT * FROM products ORDER BY sku;

sku   | name   | price | stock | status
------+--------+-------+-------+-------
SKU-1 | Widget | 999   | 100   | active
SKU-2 | Gadget | 1999  | 50    | active
SKU-3 | Gizmo  | 2999  | 0     | active
(3 rows)



## 02. INSERT variants
- Always name the columns: `INSERT INTO t (a, b) VALUES ...`. It survives later column changes.
- Many rows in one statement: `VALUES (...), (...), (...)` (faster than many single inserts).
- Skipped columns get their `DEFAULT` (or NULL if there is none).
- `INSERT INTO t DEFAULT VALUES` inserts a row made only of defaults.
- Bad vs good: `INSERT INTO t VALUES (...)` without column names breaks when someone adds a column.

In [3]:
%%sql
INSERT INTO products (sku, name, price) VALUES ('SKU-4', 'Doohickey', 499);   -- stock and status use defaults
INSERT INTO products (sku, name, price, stock) VALUES
    ('SKU-5', 'Thingamajig', 1500, 10),
    ('SKU-6', 'Whatsit', 250, 7);
SELECT * FROM products ORDER BY sku;

sku   | name        | price | stock | status
------+-------------+-------+-------+-------
SKU-1 | Widget      | 999   | 100   | active
SKU-2 | Gadget      | 1999  | 50    | active
SKU-3 | Gizmo       | 2999  | 0     | active
SKU-4 | Doohickey   | 499   | 0     | active
SKU-5 | Thingamajig | 1500  | 10    | active
SKU-6 | Whatsit     | 250   | 7     | active
(6 rows)



In [4]:
%%sql expect_error
-- Bad: wrong number of values for the named columns
INSERT INTO products (sku, name, price) VALUES ('SKU-7', 'Oops');
-- Bad: violates the CHECK constraint
INSERT INTO products (sku, name, price) VALUES ('SKU-7', 'Negative', -5);
-- Bad: duplicate primary key
INSERT INTO products (sku, name, price) VALUES ('SKU-1', 'Clone', 100);
SELECT count(*) AS still_six FROM products;

ERROR: 2 values for 3 columns

ERROR: CHECK constraint failed: price >= 0

ERROR: UNIQUE constraint failed: products.sku

still_six
---------
6        
(1 row)



## 03. INSERT OR IGNORE and other conflict clauses
- `INSERT OR IGNORE`: if the row would break a constraint (UNIQUE, PRIMARY KEY, NOT NULL, CHECK), skip just that row.
- Other forms: `OR ABORT` (default), `OR FAIL`, `OR ROLLBACK`, `OR REPLACE` (see section 09).
- Gotcha: `OR IGNORE` hides real mistakes too. It also skips rows with NOT NULL or CHECK problems, silently.
- For "insert if missing", prefer `ON CONFLICT DO NOTHING` (section 08): it only ignores uniqueness conflicts.

In [5]:
%%sql
INSERT OR IGNORE INTO products (sku, name, price) VALUES
    ('SKU-1', 'Duplicate widget', 1),   -- ignored, SKU-1 exists
    ('SKU-8', 'Newcomer', 800);         -- inserted
SELECT sku, name, price FROM products WHERE sku IN ('SKU-1', 'SKU-8') ORDER BY sku;

sku   | name     | price
------+----------+------
SKU-1 | Widget   | 999  
SKU-8 | Newcomer | 800  
(2 rows)



## 04. UPDATE with WHERE and several columns
- `UPDATE t SET a = ..., b = ... WHERE ...`: all the SET columns change together.
- No `WHERE` means EVERY row changes.
- The right-hand side sees the OLD row values, so `SET a = b, b = a` is a real swap.

In [6]:
%%sql
UPDATE products SET price = 1799, stock = 40 WHERE sku = 'SKU-2';
SELECT sku, price, stock FROM products WHERE sku = 'SKU-2';

sku   | price | stock
------+-------+------
SKU-2 | 1799  | 40   
(1 row)



In [7]:
%%sql
-- swap two columns: both right-hand sides read the old values
CREATE TABLE swap_demo (id INTEGER PRIMARY KEY, a TEXT, b TEXT);
INSERT INTO swap_demo VALUES (1, 'left', 'right');
UPDATE swap_demo SET a = b, b = a WHERE id = 1;
SELECT * FROM swap_demo;

id | a     | b   
---+-------+-----
1  | right | left
(1 row)



## 05. UPDATE with expressions, CASE and functions
- SET can compute from the current value: `stock = stock - 1`, `price = price * 110 / 100`.
- `CASE` lets one UPDATE apply different rules to different rows (instead of many UPDATEs).
- Integer maths: `price * 110 / 100` stays an integer in SQLite. Multiply first, divide last.

In [8]:
%%sql
UPDATE products
SET price = CASE
        WHEN price < 500 THEN price + 50          -- small items: flat +50 cents
        WHEN price >= 1500 THEN price * 95 / 100  -- big items: 5% off
        ELSE price
    END,
    name = upper(substr(name, 1, 1)) || substr(name, 2)
WHERE status = 'active';
SELECT sku, name, price FROM products ORDER BY sku;

sku   | name        | price
------+-------------+------
SKU-1 | Widget      | 999  
SKU-2 | Gadget      | 1709 
SKU-3 | Gizmo       | 2849 
SKU-4 | Doohickey   | 549  
SKU-5 | Thingamajig | 1425 
SKU-6 | Whatsit     | 300  
SKU-8 | Newcomer    | 800  
(7 rows)



## 06. UPDATE using a subquery
- A subquery in `SET` computes the new value from other tables.
- A subquery in `WHERE` picks which rows to update.
- Gotcha: a scalar subquery that finds nothing returns NULL, so the column becomes NULL (or fails on NOT NULL).
- See 02-Querying/03-Subqueries-And-CTEs for correlated subqueries.

In [9]:
%%sql
-- Add a column holding how many units each product has sold, then fill it with a correlated subquery
ALTER TABLE products ADD COLUMN units_sold INTEGER NOT NULL DEFAULT 0;
UPDATE products
SET units_sold = (SELECT COALESCE(SUM(o.qty), 0) FROM orders o WHERE o.sku = products.sku);
SELECT sku, units_sold FROM products ORDER BY sku;

sku   | units_sold
------+-----------
SKU-1 | 7         
SKU-2 | 1         
SKU-3 | 0         
SKU-4 | 0         
SKU-5 | 0         
SKU-6 | 0         
SKU-8 | 0         
(7 rows)



In [10]:
%%sql
-- WHERE with a subquery: mark products that nobody ever ordered
UPDATE products SET status = 'unsold'
WHERE sku NOT IN (SELECT sku FROM orders);
SELECT sku, status FROM products ORDER BY sku;

sku   | status
------+-------
SKU-1 | active
SKU-2 | active
SKU-3 | unsold
SKU-4 | unsold
SKU-5 | unsold
SKU-6 | unsold
SKU-8 | unsold
(7 rows)



## 07. UPDATE ... FROM (SQLite 3.33+)
- Lets an UPDATE join to another table or subquery, like in PostgreSQL.
- Often clearer and faster than a correlated subquery per column.
- Each target row should match at most one source row. If several match, SQLite picks one arbitrarily.
- Needs SQLite 3.33 or newer (2020).

In [11]:
%%sql
UPDATE products
SET units_sold = s.total
FROM (SELECT sku, SUM(qty) AS total FROM orders GROUP BY sku) AS s
WHERE products.sku = s.sku;
-- Only SKU-1 (7) and SKU-2 (1) have orders; same result as section 06
SELECT sku, units_sold FROM products ORDER BY sku;

sku   | units_sold
------+-----------
SKU-1 | 7         
SKU-2 | 1         
SKU-3 | 0         
SKU-4 | 0         
SKU-5 | 0         
SKU-6 | 0         
SKU-8 | 0         
(7 rows)



## 08. UPSERT: ON CONFLICT DO UPDATE / DO NOTHING
- "Insert, but if it clashes with a UNIQUE/PRIMARY KEY, do this instead."
- `ON CONFLICT (col) DO UPDATE SET ...` updates the existing row. `excluded.col` is the value you tried to insert.
- `ON CONFLICT DO NOTHING` skips the clash and still errors on other problems (NOT NULL, CHECK).
- You can refer to the old value by the table name: `stock = stock + excluded.stock`.
- Add `WHERE` to the DO UPDATE to make it conditional.
- One atomic statement: no race between "SELECT then INSERT or UPDATE".

In [12]:
%%sql
-- existing SKU-1: add stock; new SKU-9: plain insert
INSERT INTO products (sku, name, price, stock) VALUES ('SKU-1', 'Widget', 1049, 25), ('SKU-9', 'Newbie', 100, 3)
ON CONFLICT (sku) DO UPDATE SET stock = stock + excluded.stock, price = excluded.price;
SELECT sku, price, stock FROM products WHERE sku IN ('SKU-1', 'SKU-9') ORDER BY sku;

sku   | price | stock
------+-------+------
SKU-1 | 1049  | 125  
SKU-9 | 100   | 3    
(2 rows)



In [13]:
%%sql
-- DO NOTHING: keep the first email, ignore the repeat. Conditional DO UPDATE: only accept a lower price.
INSERT INTO customers (name, email, city) VALUES ('Asha Again', 'asha@example.com', 'Mumbai')
ON CONFLICT (email) DO NOTHING;
INSERT INTO products (sku, name, price) VALUES ('SKU-9', 'Newbie', 150)
ON CONFLICT (sku) DO UPDATE SET price = excluded.price WHERE excluded.price < products.price;   -- 150 > 100, so no change
INSERT INTO products (sku, name, price) VALUES ('SKU-9', 'Newbie', 90)
ON CONFLICT (sku) DO UPDATE SET price = excluded.price WHERE excluded.price < products.price;   -- 90 < 100, updated
SELECT sku, price FROM products WHERE sku = 'SKU-9';
SELECT id, name, city FROM customers ORDER BY id;

sku   | price
------+------
SKU-9 | 90   
(1 row)

id | name   | city 
---+--------+------
1  | Asha   | Pune 
2  | Ben    | Delhi
3  | Chitra | Pune 
(3 rows)



## 09. INSERT OR REPLACE is not an UPSERT
- `INSERT OR REPLACE` (and `REPLACE INTO`) on a conflict DELETES the old row, then INSERTS a new one.
- Columns you did not mention go back to their defaults. The old values are lost.
- The delete is real: `ON DELETE CASCADE` children are removed (when foreign keys are ON). The rowid can change.
- `ON CONFLICT DO UPDATE` modifies the row in place: other columns and child rows stay.
- Rule of thumb: use UPSERT, avoid REPLACE on tables that have children.

In [14]:
%%sql
-- Asha (id 1) has two orders. Re-insert her with REPLACE...
INSERT OR REPLACE INTO customers (id, name, email, city) VALUES (1, 'Asha', 'asha@example.com', 'Pune');
SELECT 'orders left for customer 1' AS check_name, count(*) AS n FROM orders WHERE customer_id = 1;

check_name                 | n
---------------------------+--
orders left for customer 1 | 0
(1 row)



In [15]:
%%sql
-- ...her two orders are gone (cascade). Rebuild them, then do the same with UPSERT.
INSERT INTO orders (customer_id, sku, qty, placed_on) VALUES (1, 'SKU-1', 2, '2025-01-05'), (1, 'SKU-2', 1, '2025-01-09');
INSERT INTO customers (id, name, email, city) VALUES (1, 'Asha', 'asha@example.com', 'Goa')
ON CONFLICT (id) DO UPDATE SET city = excluded.city;
SELECT 'orders after upsert' AS check_name, count(*) AS n FROM orders WHERE customer_id = 1;
SELECT id, name, city FROM customers WHERE id = 1;

check_name          | n
--------------------+--
orders after upsert | 2
(1 row)

id | name | city
---+------+-----
1  | Asha | Goa 
(1 row)



## 10. DELETE with WHERE and subqueries
- `DELETE FROM t WHERE ...` removes matching rows. No WHERE removes all rows (the table stays).
- Subqueries in WHERE delete rows based on other tables.
- Foreign keys matter: deleting a parent that still has children fails, unless the key says `ON DELETE CASCADE` / `SET NULL`.
- `NOT IN` plus a NULL in the subquery matches nothing. Use `NOT EXISTS` to be safe.

In [16]:
%%sql
-- delete customers who never ordered (NOT EXISTS is NULL-safe)
DELETE FROM customers
WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = customers.id);
SELECT id, name FROM customers ORDER BY id;

id | name
---+-----
1  | Asha
2  | Ben 
(2 rows)



In [17]:
%%sql expect_error
-- A product that has orders cannot be deleted (orders.sku REFERENCES products without CASCADE)
DELETE FROM products WHERE sku = 'SKU-1';
SELECT count(*) AS products_left FROM products;

ERROR: FOREIGN KEY constraint failed

products_left
-------------
8            
(1 row)



## 11. Safe-delete habits
- Step 1: run the `SELECT` with the same `WHERE`. Look at the rows and the count.
- Step 2: do it inside a transaction: `BEGIN; DELETE ...; SELECT ...; ROLLBACK;` to preview, then `COMMIT` when happy.
- Step 3: check `changes()` right after. If you expected 3 and got 3000, ROLLBACK.
- Prefer soft deletes (section 17) for business data you might need again.
- See 02-Transactions for BEGIN/COMMIT/ROLLBACK details.

In [18]:
%%sql
-- 1) preview
SELECT sku, name, stock FROM products WHERE stock = 0 ORDER BY sku;

sku   | name      | stock
------+-----------+------
SKU-3 | Gizmo     | 0    
SKU-4 | Doohickey | 0    
SKU-8 | Newcomer  | 0    
(3 rows)



In [19]:
%%sql
-- 2) try it in a transaction, check, then roll back (or commit)
BEGIN;
DELETE FROM products WHERE stock = 0;
SELECT changes() AS rows_deleted;
SELECT count(*) AS products_inside_txn FROM products;
ROLLBACK;
SELECT count(*) AS products_after_rollback FROM products;

rows_deleted
------------
3           
(1 row)

products_inside_txn
-------------------
5                  
(1 row)

products_after_rollback
-----------------------
8                      
(1 row)



## 12. RETURNING (SQLite 3.35+)
- `INSERT`, `UPDATE`, `DELETE` can end with `RETURNING cols`: you get the affected rows back as a result set.
- Great for fetching generated ids or the new values without a second SELECT.
- For UPDATE it shows the NEW values. For DELETE it shows what was removed.
- The order of the returned rows is not guaranteed, so do not rely on it.

In [20]:
%%sql
INSERT INTO customers (name, email, city) VALUES ('Dev', 'dev@example.com', 'Pune') RETURNING id, name;

id | name
---+-----
3  | Dev 
(1 row)



In [21]:
%%sql
UPDATE products SET stock = stock - 3 WHERE sku = 'SKU-1' RETURNING sku, stock AS new_stock;

sku   | new_stock
------+----------
SKU-1 | 122      
(1 row)



In [22]:
%%sql
DELETE FROM customers WHERE email = 'dev@example.com' RETURNING id, name, email;

id | name | email          
---+------+----------------
3  | Dev  | dev@example.com
(1 row)



## 13. changes() and last_insert_rowid()
- `changes()`: number of rows changed by the most recent INSERT/UPDATE/DELETE on this connection.
- `total_changes()`: running total since the connection opened.
- `last_insert_rowid()`: the rowid (or INTEGER PRIMARY KEY) of the last inserted row on this connection.
- Gotcha: they describe the last statement, so call them right after. A SELECT in between does not reset `changes()`, but another write does.
- Gotcha: `last_insert_rowid()` after `INSERT OR IGNORE` that skipped is the PREVIOUS value. Use RETURNING instead.

In [23]:
%%sql
INSERT INTO customers (name, email, city) VALUES ('Esha', 'esha@example.com', 'Goa');
SELECT last_insert_rowid() AS new_id;
UPDATE products SET stock = stock + 1 WHERE stock < 10;
SELECT changes() AS rows_touched;

new_id
------
3     
(1 row)

rows_touched
------------
5           
(1 row)



## 14. Bulk copy: INSERT ... SELECT and CREATE TABLE AS SELECT
- `INSERT INTO target (cols) SELECT ... FROM source`: copy or transform rows in one statement.
- `CREATE TABLE new AS SELECT ...`: makes a table from a query. Handy for snapshots and reports.
- Gotcha: CTAS copies data and column names only. No PRIMARY KEY, NOT NULL, defaults, or indexes. Create the table yourself if you need those.
- Always name the columns on both sides so the order cannot go wrong.

In [24]:
%%sql
CREATE TABLE products_archive (sku TEXT PRIMARY KEY, name TEXT NOT NULL, price INTEGER NOT NULL, archived_on TEXT NOT NULL);
INSERT INTO products_archive (sku, name, price, archived_on)
SELECT sku, name, price, '2025-03-01' FROM products WHERE status = 'unsold';
SELECT * FROM products_archive ORDER BY sku;

sku   | name        | price | archived_on
------+-------------+-------+------------
SKU-3 | Gizmo       | 2849  | 2025-03-01 
SKU-4 | Doohickey   | 549   | 2025-03-01 
SKU-5 | Thingamajig | 1425  | 2025-03-01 
SKU-6 | Whatsit     | 300   | 2025-03-01 
SKU-8 | Newcomer    | 800   | 2025-03-01 
(5 rows)



In [25]:
%%sql
CREATE TABLE sales_snapshot AS
SELECT p.sku, p.name, COALESCE(SUM(o.qty), 0) AS qty_sold
FROM products p LEFT JOIN orders o ON o.sku = p.sku
GROUP BY p.sku, p.name;
SELECT * FROM sales_snapshot ORDER BY sku;

sku   | name        | qty_sold
------+-------------+---------
SKU-1 | Widget      | 7       
SKU-2 | Gadget      | 1       
SKU-3 | Gizmo       | 0       
SKU-4 | Doohickey   | 0       
SKU-5 | Thingamajig | 0       
SKU-6 | Whatsit     | 0       
SKU-8 | Newcomer    | 0       
SKU-9 | Newbie      | 0       
(8 rows)



In [26]:
%%sql
-- what CTAS did and did not keep: no primary key, no constraints
SELECT name, type, "notnull" AS not_null, pk FROM pragma_table_info('sales_snapshot') ORDER BY cid;

name     | type | not_null | pk
---------+------+----------+---
sku      | TEXT | 0        | 0 
name     | TEXT | 0        | 0 
qty_sold |      | 0        | 0 
(3 rows)



## 15. Emptying a table: the TRUNCATE equivalent
- SQLite has no `TRUNCATE TABLE`. Use `DELETE FROM t` (no WHERE). SQLite optimises this to a fast truncate when it can.
- `DROP TABLE` removes the table itself; `DELETE FROM` keeps the structure.
- Auto-numbering: with plain `INTEGER PRIMARY KEY` the next id restarts from max+1. With `AUTOINCREMENT`, the counter lives in `sqlite_sequence`; reset it with `DELETE FROM sqlite_sequence WHERE name = 't'`.

In [27]:
%%sql
CREATE TABLE log_auto (id INTEGER PRIMARY KEY AUTOINCREMENT, msg TEXT);
INSERT INTO log_auto (msg) VALUES ('a'), ('b'), ('c');
DELETE FROM log_auto;
INSERT INTO log_auto (msg) VALUES ('after delete');
SELECT * FROM log_auto;   -- id is 4, AUTOINCREMENT never reuses numbers
DELETE FROM log_auto;
DELETE FROM sqlite_sequence WHERE name = 'log_auto';
INSERT INTO log_auto (msg) VALUES ('after reset');
SELECT * FROM log_auto;   -- id is 1 again

id | msg         
---+-------------
4  | after delete
(1 row)

id | msg        
---+------------
1  | after reset
(1 row)



## 16. ALTER TABLE: what works and what does not
- Works: `RENAME TO`, `RENAME COLUMN` (3.25+), `ADD COLUMN`, `DROP COLUMN` (3.35+).
- `ADD COLUMN` limits: cannot be PRIMARY KEY or UNIQUE; a NOT NULL column needs a non-NULL DEFAULT; the default must be a constant.
- `DROP COLUMN` fails if the column is a PRIMARY KEY, UNIQUE, indexed, used in a CHECK, or referenced by a foreign key, view or trigger.
- Cannot do: change a column type, add or remove constraints on an existing column, reorder columns.
- For those, use the rebuild recipe: create new table, copy rows, drop old, rename new (section 18).

In [28]:
%%sql
ALTER TABLE customers ADD COLUMN phone TEXT;
ALTER TABLE customers RENAME COLUMN phone TO mobile;
ALTER TABLE customers DROP COLUMN mobile;
SELECT name FROM pragma_table_info('customers') ORDER BY cid;

name 
-----
id   
name 
email
city 
(4 rows)



In [29]:
%%sql expect_error
-- NOT NULL without a default: refused
ALTER TABLE customers ADD COLUMN signup_date TEXT NOT NULL;
-- UNIQUE column cannot be added this way
ALTER TABLE customers ADD COLUMN nick TEXT UNIQUE;
-- cannot drop a column that is part of the primary key
ALTER TABLE products DROP COLUMN sku;

ERROR: Cannot add a NOT NULL column with default value NULL

ERROR: Cannot add a UNIQUE column

ERROR: cannot drop PRIMARY KEY column: "sku"



## 17. Soft deletes
- Instead of removing the row, mark it: `deleted_at TEXT` (NULL = alive) or `is_deleted INTEGER`.
- Pros: undo is easy, history stays, foreign keys keep working.
- Cons: EVERY query must remember `WHERE deleted_at IS NULL`. Hide that in a view. UNIQUE columns still collide with soft-deleted rows.
- Good for orders, users, documents. Bad when law requires real erasure (privacy).
- See 05-DDL-And-Integrity/02-Views-And-Triggers for views.

In [30]:
%%sql
ALTER TABLE customers ADD COLUMN deleted_at TEXT;
CREATE VIEW active_customers AS SELECT id, name, city FROM customers WHERE deleted_at IS NULL;
UPDATE customers SET deleted_at = '2025-03-01' WHERE name = 'Ben';
SELECT * FROM active_customers ORDER BY id;
SELECT id, name, deleted_at FROM customers ORDER BY id;   -- Ben is still there

id | name | city
---+------+-----
1  | Asha | Goa 
3  | Esha | Goa 
(2 rows)

id | name | deleted_at
---+------+-----------
1  | Asha | NULL      
2  | Ben  | 2025-03-01
3  | Esha | NULL      
(3 rows)



## 18. Rebuilding a table to change what ALTER cannot
- Recipe: `BEGIN`, create `new_t` with the shape you want, `INSERT INTO new_t SELECT ...`, `DROP TABLE t`, `ALTER TABLE new_t RENAME TO t`, recreate indexes/triggers/views, `COMMIT`.
- It is safe because DDL is transactional in SQLite.
- Example below: add a CHECK and a proper type to `sales_snapshot`, which CTAS left unconstrained.

In [31]:
%%sql
BEGIN;
CREATE TABLE sales_snapshot_new (
    sku      TEXT PRIMARY KEY,
    name     TEXT NOT NULL,
    qty_sold INTEGER NOT NULL CHECK (qty_sold >= 0)
);
INSERT INTO sales_snapshot_new SELECT sku, name, qty_sold FROM sales_snapshot;
DROP TABLE sales_snapshot;
ALTER TABLE sales_snapshot_new RENAME TO sales_snapshot;
COMMIT;
SELECT * FROM sales_snapshot ORDER BY sku;
SELECT name, "notnull" AS not_null, pk FROM pragma_table_info('sales_snapshot') ORDER BY cid;

sku   | name        | qty_sold
------+-------------+---------
SKU-1 | Widget      | 7       
SKU-2 | Gadget      | 1       
SKU-3 | Gizmo       | 0       
SKU-4 | Doohickey   | 0       
SKU-5 | Thingamajig | 0       
SKU-6 | Whatsit     | 0       
SKU-8 | Newcomer    | 0       
SKU-9 | Newbie      | 0       
(8 rows)

name     | not_null | pk
---------+----------+---
sku      | 0        | 1 
name     | 1        | 0 
qty_sold | 1        | 0 
(3 rows)



## 19. Putting it together: a stock receipt
- Real-world pattern: receive a delivery, bump stock for existing SKUs, create new ones, and report what changed, all with set-based statements.
- Compare: a Python loop with a SELECT + INSERT/UPDATE per row is slower and has race conditions.

In [32]:
%%sql
CREATE TABLE delivery (sku TEXT, name TEXT, qty INTEGER, price INTEGER);
INSERT INTO delivery VALUES ('SKU-2', 'Gadget', 20, 1799), ('SKU-20', 'Fresh item', 15, 700), ('SKU-4', 'Doohickey', 5, 499);
-- (WHERE true is needed: it stops SQLite reading ON CONFLICT as part of a join clause)
INSERT INTO products (sku, name, price, stock)
SELECT sku, name, price, qty FROM delivery WHERE true
ON CONFLICT (sku) DO UPDATE SET stock = stock + excluded.stock
RETURNING sku, stock;

sku    | stock
-------+------
SKU-2  | 60   
SKU-20 | 15   
SKU-4  | 6    
(3 rows)



In [33]:
%%sql
SELECT sku, name, price, stock, status FROM products ORDER BY sku;

sku    | name        | price | stock | status
-------+-------------+-------+-------+-------
SKU-1  | Widget      | 1049  | 122   | active
SKU-2  | Gadget      | 1709  | 60    | active
SKU-20 | Fresh item  | 700   | 15    | active
SKU-3  | Gizmo       | 2849  | 1     | unsold
SKU-4  | Doohickey   | 549   | 6     | unsold
SKU-5  | Thingamajig | 1425  | 10    | unsold
SKU-6  | Whatsit     | 300   | 8     | unsold
SKU-8  | Newcomer    | 800   | 1     | unsold
SKU-9  | Newbie      | 90    | 4     | active
(9 rows)

